In [0]:
from pyspark.sql.functions import col, date_sub, date_add, count, avg, round, collect_list, struct

In [0]:

dbutils.widgets.text("gold_catalog","dbr_dev")
dbutils.widgets.text("gold_schema","artemzharkov10_gold")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

In [0]:
df_anomalies = spark.read.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.finance_bitcoin_anomaly_gold")
df_news = spark.read.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.bitcoin_news_gold")

df_same_day = df_anomalies.join(
    df_news,
    col("AnomalyDate") == col("NewsDateClean"), 
    "inner"
)

In [0]:
df_mart = (
    df_same_day
    .groupBy(
        "AnomalyDate",          
        "PriceOnAnomalyDay",    
        "DeviationPercentage"   
    )
    .agg(
        count("URL").alias("NewsCountOnAnomalyDay"),           
        round(avg("AverageTone"), 2).alias("AverageToneOnDay"),

    )
    .orderBy("AnomalyDate")
)

In [0]:
(df_mart.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.bitcoin_same_day_mart"))

In [0]:
display(df_mart.limit(10))